# 리포트 62 — CPI 를 늘리면 세 파형 모두 블라인드율이 내려간다

> ### 한 일
> **0-도플러 가드가 지우는 헤딩 비율을 CPI 축에서 스윕해, 5G 가 치르는 대가가 적분시간으로 줄어드는 부분과 배수로 남는 부분을 갈랐다.**

### 결과
1. 5G 의 눈먼 헤딩 비율은 CPI 0.1 s ⟨outputs/cpi_guard_sweep.json : equal_cpi_penalty[0].T_cpi_s⟩ 에서 0.636 ⟨outputs/cpi_guard_sweep.json : verdict.artifact.blind_hard_same_cpi⟩, CPI 0.2 s ⟨outputs/cpi_guard_sweep.json : equal_cpi_penalty[1].T_cpi_s⟩ 에서 0.303 ⟨outputs/cpi_guard_sweep.json : verdict.artifact.blind_hard_at_200ms⟩ 로 내려간다.
2. WiFi 대비 배수는 CPI 전 구간에서 12.1 ⟨outputs/cpi_guard_sweep.json : equal_cpi_penalty[0].ratio_G1_over_W1⟩~19.0 배 ⟨outputs/cpi_guard_sweep.json : equal_cpi_penalty[3].ratio_G1_over_W1⟩ 로 남는다 — 이것이 이 대가를 구조로 만드는 첫 번째 사실이다.
3. 도플러 축을 지우는 기구는 둘이다 — 표본화 쪽은 가드가 접힘 축 전체를 덮는 구조식 `guard_hz = g*PRF/M >= PRF/2  <=>  M <= 2g  <=>  T_cpi <= 2g/PRF ⟨outputs/cpi_guard_sweep.json : structural.formula⟩` 이고, 진폭 쪽은 짧은 CPI 에서 가드가 도플러 진폭을 덮는 파형 공통 현상이다.
4. 5G 의 alias 비율 0.861 ⟨outputs/cpi_guard_sweep.json : verdict.structural.s2_alias_floor.alias_frac_G1⟩ 는 CPI 와 무관한 상수이고, WiFi·LTE 는 0.000 ⟨outputs/cpi_guard_sweep.json : verdict.structural.s2_alias_floor.alias_frac_W1⟩ 다.

### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 가드 규약 | 가드 반폭 = g빈 × PRF/M 이고 g 는 검출기 적용값 1.5빈과 선언값 2.5빈 둘 다 잰다 |
| 격자 | 헤딩 720 ⟨outputs/cpi_guard_sweep.json : meta.psi_n_fine⟩점 · 표적 속도 5 m/s ⟨outputs/cpi_guard_sweep.json : meta.geometry.speed_ms⟩ — `benchmark/cpi_guard_sweep.py` |
| 두 기구 | 표본화(접힘 축 전체를 가드가 덮는가)와 진폭(도플러 진폭이 가드 안에 드는가)을 따로 센다 |

### 재현

```bash
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/cpi_guard_sweep.py
PYTHONPATH=src ~/.venvs/py312/bin/python src/build_part10_results.py
```

| | |
|---|---|
| 출력 | `outputs/cpi_guard_sweep.json` |
| 소요 | 3.7 s ⟨outputs/report05_derived.json : runtime.cpi_guard_sweep_s⟩ |

---

## 5G 의 상시기준 대가 — CPI 스윕

5G 의 상시 기준신호(SSB)는 20 ms 주기라 PRF 50 Hz 를 준다. 도플러 축을 지우는 기구는 둘이다 — **A. 표본화**: `guard_hz = g*PRF/M >= PRF/2  <=>  M <= 2g  <=>  T_cpi <= 2g/PRF ⟨outputs/cpi_guard_sweep.json : structural.formula⟩` 로 가드가 접힘 축 전체를 덮는다(반송파·속도·거리 무관). **B. 진폭**: 짧은 CPI 에서 가드가 도플러 진폭을 덮는다(파형 공통). 1.5빈 규약에서 LTE 도 CPI ≤ 0.024 s ⟨outputs/cpi_guard_sweep.json : structural.two_mechanisms.observed.L1.hard.T_max_total_blind_s⟩ 에서 전 헤딩 블라인드가 된다.

| 모드 | PRF | CPI 0.1 s 의 M | 접힘 축 ± | 블라인드(1.5빈) | 블라인드(2.5빈) |
|---|---|---|---|---|---|
| WiFi | 1000 Hz ⟨outputs/cpi_guard_sweep.json : waveform_facts.W1.prf_hz⟩ | 100 ⟨outputs/cpi_guard_sweep.json : anchor.reproduction.W1.M⟩ | 500.0 Hz ⟨outputs/cpi_guard_sweep.json : anchor.reproduction.W1.fold_half_hz⟩ | 0.028 ⟨outputs/cpi_guard_sweep.json : anchor.reproduction.W1.blind_hard⟩ | 0.083 ⟨outputs/cpi_guard_sweep.json : anchor.reproduction.W1.blind_declared⟩ |
| LTE | 1000 Hz ⟨outputs/cpi_guard_sweep.json : waveform_facts.L1.prf_hz⟩ | 100 ⟨outputs/cpi_guard_sweep.json : anchor.reproduction.L1.M⟩ | 500.0 Hz ⟨outputs/cpi_guard_sweep.json : anchor.reproduction.L1.fold_half_hz⟩ | 0.139 ⟨outputs/cpi_guard_sweep.json : anchor.reproduction.L1.blind_hard⟩ | 0.250 ⟨outputs/cpi_guard_sweep.json : anchor.reproduction.L1.blind_declared⟩ |
| 5G | 50 Hz ⟨outputs/cpi_guard_sweep.json : waveform_facts.G1.prf_hz⟩ | 5 ⟨outputs/cpi_guard_sweep.json : anchor.reproduction.G1.M⟩ | 25.0 Hz ⟨outputs/cpi_guard_sweep.json : anchor.reproduction.G1.fold_half_hz⟩ | 0.639 ⟨outputs/cpi_guard_sweep.json : anchor.reproduction.G1.blind_hard⟩ | 1.000 ⟨outputs/cpi_guard_sweep.json : anchor.reproduction.G1.blind_declared⟩ |

5G 의 커버리지 0 은 선언가드 2.5빈 · CPI ≤ 0.10 s ⟨outputs/cpi_guard_sweep.json : structural.by_mode.G1.T_max_total_blind_declared_s⟩ 에서 성립한다. 검출기가 적용하는 1.5빈 규약의 경계는 0.06 s ⟨outputs/cpi_guard_sweep.json : structural.by_mode.G1.T_max_total_blind_hard_s⟩ 이고, CPI 0.1 s ⟨outputs/cpi_guard_sweep.json : equal_cpi_penalty[0].T_cpi_s⟩ 의 블라인드율은 0.636 ⟨outputs/cpi_guard_sweep.json : verdict.artifact.blind_hard_same_cpi⟩ 다.

CPI 를 늘리면 세 파형 모두 블라인드율이 내려간다. 5G 가 치르는 **배수**는 그대로 남는다 — 이것이 이 대가를 구조로 만드는 첫 번째 사실이다.

| CPI | WiFi | LTE | 5G | 5G/WiFi | 5G/LTE |
|---|---|---|---|---|---|
| 0.1 s ⟨outputs/cpi_guard_sweep.json : equal_cpi_penalty[0].T_cpi_s⟩ | 0.053 ⟨outputs/cpi_guard_sweep.json : equal_cpi_penalty[0].blind_hard_W1⟩ | 0.158 ⟨outputs/cpi_guard_sweep.json : equal_cpi_penalty[0].blind_hard_L1⟩ | 0.636 ⟨outputs/cpi_guard_sweep.json : equal_cpi_penalty[0].blind_hard_G1⟩ | 12.1 배 ⟨outputs/cpi_guard_sweep.json : equal_cpi_penalty[0].ratio_G1_over_W1⟩ | 4.0 배 ⟨outputs/cpi_guard_sweep.json : equal_cpi_penalty[0].ratio_G1_over_L1⟩ |
| 0.2 s ⟨outputs/cpi_guard_sweep.json : equal_cpi_penalty[1].T_cpi_s⟩ | 0.025 ⟨outputs/cpi_guard_sweep.json : equal_cpi_penalty[1].blind_hard_W1⟩ | 0.081 ⟨outputs/cpi_guard_sweep.json : equal_cpi_penalty[1].blind_hard_L1⟩ | 0.303 ⟨outputs/cpi_guard_sweep.json : equal_cpi_penalty[1].blind_hard_G1⟩ | 12.1 배 ⟨outputs/cpi_guard_sweep.json : equal_cpi_penalty[1].ratio_G1_over_W1⟩ | 3.8 배 ⟨outputs/cpi_guard_sweep.json : equal_cpi_penalty[1].ratio_G1_over_L1⟩ |
| 0.5 s ⟨outputs/cpi_guard_sweep.json : equal_cpi_penalty[2].T_cpi_s⟩ | 0.008 ⟨outputs/cpi_guard_sweep.json : equal_cpi_penalty[2].blind_hard_W1⟩ | 0.031 ⟨outputs/cpi_guard_sweep.json : equal_cpi_penalty[2].blind_hard_L1⟩ | 0.119 ⟨outputs/cpi_guard_sweep.json : equal_cpi_penalty[2].blind_hard_G1⟩ | 14.3 배 ⟨outputs/cpi_guard_sweep.json : equal_cpi_penalty[2].ratio_G1_over_W1⟩ | 3.9 배 ⟨outputs/cpi_guard_sweep.json : equal_cpi_penalty[2].ratio_G1_over_L1⟩ |
| 1.0 s ⟨outputs/cpi_guard_sweep.json : equal_cpi_penalty[3].T_cpi_s⟩ | 0.003 ⟨outputs/cpi_guard_sweep.json : equal_cpi_penalty[3].blind_hard_W1⟩ | 0.014 ⟨outputs/cpi_guard_sweep.json : equal_cpi_penalty[3].blind_hard_L1⟩ | 0.053 ⟨outputs/cpi_guard_sweep.json : equal_cpi_penalty[3].blind_hard_G1⟩ | 19.0 배 ⟨outputs/cpi_guard_sweep.json : equal_cpi_penalty[3].ratio_G1_over_W1⟩ | 3.8 배 ⟨outputs/cpi_guard_sweep.json : equal_cpi_penalty[3].ratio_G1_over_L1⟩ |
| 2.0 s ⟨outputs/cpi_guard_sweep.json : equal_cpi_penalty[4].T_cpi_s⟩ | 0.003 ⟨outputs/cpi_guard_sweep.json : equal_cpi_penalty[4].blind_hard_W1⟩ | 0.008 ⟨outputs/cpi_guard_sweep.json : equal_cpi_penalty[4].blind_hard_L1⟩ | 0.031 ⟨outputs/cpi_guard_sweep.json : equal_cpi_penalty[4].blind_hard_G1⟩ | 11.0 배 ⟨outputs/cpi_guard_sweep.json : equal_cpi_penalty[4].ratio_G1_over_W1⟩ | 3.7 배 ⟨outputs/cpi_guard_sweep.json : equal_cpi_penalty[4].ratio_G1_over_L1⟩ |

두 번째 사실은 접힘이다 — 5G 의 alias 비율 0.861 ⟨outputs/cpi_guard_sweep.json : verdict.structural.s2_alias_floor.alias_frac_G1⟩ 는 적분시간이 아니라 표본화율의 성질이라 CPI 와 무관한 상수이고, WiFi·LTE 는 0.000 ⟨outputs/cpi_guard_sweep.json : verdict.structural.s2_alias_floor.alias_frac_W1⟩ 다.

## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| CPI 를 0.1 s 에서 1.0 s 까지 정본 solve 에 넣어 R90(CPI) 를 낸다 | 이 편의 커버리지 회복이 거리 축에서도 확정된다 | `benchmark/cpi_guard_sweep.py` → `src/experiment_freespace_range.py` |